# Use the SKSurrogate API from Python

This notebook uses Python's standard-library HTTP client, so it does not add a client dependency. It uploads reproducible synthetic train/validation partitions, profiles the registered training data, submits a bounded pipeline search, evaluates the selected bundle, and downloads its run configuration and Markdown report.

Start the API in a separate terminal from the repository root with `python -m uvicorn api.main:app --host 127.0.0.1 --port 8000`. Set `SKSURROGATE_API_URL` if the server uses another address and `SKSURROGATE_API_KEY` if authentication is enabled. The API stores uploaded datasets, jobs, and bundles under its configured storage root; use a disposable task name and non-sensitive data for this example.

In [ ]:
import json
import os
import time
import uuid
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE_URL = os.environ.get("SKSURROGATE_API_URL", "http://127.0.0.1:8000").rstrip("/")
API_KEY = os.environ.get("SKSURROGATE_API_KEY")

def api_request(path, *, method="GET", payload=None, upload=None, timeout=60):
    """Send JSON or a single CSV multipart upload; raise with server errors intact."""
    headers = {"Accept": "application/json"}
    if API_KEY:
        headers["X-API-Key"] = API_KEY
    if upload is not None:
        boundary = "----sksurrogate-" + uuid.uuid4().hex
        fields, filename, content = upload
        parts = []
        for name, value in fields.items():
            parts.extend([
                f"--{boundary}\r\n".encode(),
                f'Content-Disposition: form-data; name="{name}"\r\n\r\n'.encode(),
                str(value).encode(),
                b"\r\n",
            ])
        parts.extend([
            f"--{boundary}\r\n".encode(),
            f'Content-Disposition: form-data; name="file"; filename="{filename}"\r\n'.encode(),
            b"Content-Type: text/csv\r\n\r\n",
            content,
            b"\r\n",
            f"--{boundary}--\r\n".encode(),
        ])
        data = b"".join(parts)
        headers["Content-Type"] = f"multipart/form-data; boundary={boundary}"
    elif payload is not None:
        data = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"
    else:
        data = None

    request = Request(BASE_URL + path, data=data, headers=headers, method=method)
    try:
        with urlopen(request, timeout=timeout) as response:
            body = response.read()
            content_type = response.headers.get("Content-Type", "")
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"API returned HTTP {exc.code}: {detail}") from exc
    except URLError as exc:
        raise RuntimeError(f"Could not reach SKSurrogate API at {BASE_URL}: {exc}") from exc
    if "application/json" in content_type:
        return json.loads(body)
    return body

health = api_request("/api/health")
print("API health:", health)

## Prepare and register data

A fixed seed makes both partitions reproducible. Uploading the validation partition separately keeps it out of the training search.

In [ ]:
import io

import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

SEED = 7
X, y = make_classification(
    n_samples=240,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    random_state=SEED,
)
frame = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
frame["target"] = y
train, validation = train_test_split(
    frame, test_size=0.25, random_state=SEED, stratify=frame["target"]
)
task_name = os.environ.get("SKSURROGATE_TASK_NAME", "notebook-" + time.strftime("%Y%m%d-%H%M%S"))

def csv_upload(partition_frame, partition):
    content = partition_frame.to_csv(index=False).encode("utf-8")
    return ({"target": "target", "partition": partition}, f"{partition}.csv", content)

registered = {}
for partition, partition_frame in (("train", train), ("validation", validation)):
    registered[partition] = api_request(
        f"/api/datasets/{task_name}/register",
        method="POST",
        upload=csv_upload(partition_frame, partition),
    )
print({name: item["dataset_fingerprint"] for name, item in registered.items()})

## Profile, search, and evaluate

The job endpoints are asynchronous. This notebook polls the persisted job record and raises on failure or timeout instead of presenting a failed job as successful.

In [ ]:
profile = api_request(
    f"/api/datasets/{task_name}/profile",
    method="POST",
    payload={"partition": "train", "max_rows": 500, "random_state": SEED},
)
print("Profile rows/features:", profile["sampled_rows"], profile["feature_count"])

search = api_request(
    f"/api/experiments/{task_name}/run",
    method="POST",
    payload={
        "config": {
            "sklearn.linear_model.LogisticRegression": {
                "C": {"type": "real", "low": 0.1, "high": 1.0}
            }
        },
        "length": 1,
        "max_generation": 1,
        "num_parents": 1,
        "train_partition": "train",
        "scoring": "accuracy",
        "random_state": SEED,
    },
)

def wait_for_job(job_id, *, timeout=300, interval=1):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        job = api_request(f"/api/jobs/{job_id}")
        if job["status"] == "completed":
            return job["result"]
        if job["status"] == "failed":
            raise RuntimeError(f"Job {job_id} failed:\n{job.get('error')}")
        time.sleep(interval)
    raise TimeoutError(f"Job {job_id} did not complete within {timeout} seconds")

search_result = wait_for_job(search["job_id"])
model_version = search_result["model_version"]
print("Selected model version:", model_version)

evaluation = api_request(
    f"/api/evaluation/{task_name}/nested-cv",
    method="POST",
    payload={
        "model_version": model_version,
        "inner_cv": 2,
        "outer_cv": 2,
        "train_partition": "train",
        "validation_partition": "validation",
        "scoring": "accuracy",
        "repeats": 2,
        "confidence_level": 0.95,
    },
)
evaluation_result = wait_for_job(evaluation["job_id"])
print("Repeated outer scores:", evaluation_result["repeated_outer_scores"])
print("Confidence interval:", evaluation_result["confidence_interval"])

## Export the run record

The JSON configuration and Markdown report are generated from the saved model bundle. They retain partition/model lineage and the evaluation uncertainty recorded above.

In [ ]:
run_config = api_request(
    f"/api/bundles/{task_name}/{model_version}/run-config"
)
report_bytes = api_request(
    f"/api/bundles/{task_name}/{model_version}/report.md"
)
print("Training fingerprint:", run_config["lineage"]["train_partition_fingerprint"])
print(report_bytes.decode("utf-8")[:2000])